# 04 - Evaluacion sobre Test

Carga ambos modelos entrenados y evalua sobre `splits/test/`. Genera matrices de confusion y reporta F1 por clase.


In [ ]:
!pip install -q tensorflow scikit-learn matplotlib seaborn

In [ ]:
import tensorflow as tf
import numpy as np
import json
from pathlib import Path
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, f1_score, precision_score, recall_score,
)
import matplotlib.pyplot as plt
import seaborn as sns

OUT = Path("./outputs")
SPLIT = Path("./splits")


class AsymmetricLoss(tf.keras.losses.Loss):
    def __init__(self, gamma_neg=4.0, gamma_pos=0.0, clip=0.05, label_smoothing=0.05,
                 name="asymmetric_loss", **kwargs):
        super().__init__(name=name, **kwargs)
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip
        self.label_smoothing = label_smoothing
        self.eps = 1e-8

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.cast(y_pred, tf.float32)
        num_classes = tf.cast(tf.shape(y_true)[-1], tf.float32)
        y_true_s = y_true * (1.0 - self.label_smoothing) + self.label_smoothing / num_classes
        xs_pos = y_pred
        xs_neg = 1.0 - y_pred
        if self.clip > 0:
            xs_neg = tf.clip_by_value(xs_neg + self.clip, 0.0, 1.0)
        log_pos = tf.math.log(tf.clip_by_value(xs_pos, self.eps, 1.0))
        log_neg = tf.math.log(tf.clip_by_value(xs_neg, self.eps, 1.0))
        loss_pos = y_true_s * log_pos
        loss_neg = (1.0 - y_true_s) * log_neg
        if self.gamma_pos > 0:
            loss_pos = loss_pos * tf.pow(1.0 - xs_pos, self.gamma_pos)
        if self.gamma_neg > 0:
            loss_neg = loss_neg * tf.pow(1.0 - xs_neg, self.gamma_neg)
        return -tf.reduce_sum(loss_pos + loss_neg, axis=-1)

    def get_config(self):
        base = super().get_config()
        base.update({
            "gamma_neg": self.gamma_neg,
            "gamma_pos": self.gamma_pos,
            "clip": self.clip,
            "label_smoothing": self.label_smoothing,
        })
        return base


_custom = {"AsymmetricLoss": AsymmetricLoss}

m1 = tf.keras.models.load_model(OUT / "model1_binary.keras")
m2 = tf.keras.models.load_model(OUT / "model2_pathogen.keras", custom_objects=_custom)
print("Modelos cargados OK")

In [ ]:
import sys
sys.path.insert(0, "..")

import albumentations as A
import numpy as np
from pathlib import Path
from PIL import Image
from tensorflow.keras.utils import Sequence
from tensorflow.keras.applications.efficientnet import preprocess_input

VAL_AUG = A.Compose([])

class LeafSequence(Sequence):
    def __init__(self, directory, img_size=(224, 224), batch_size=32, class_mode="binary"):
        self.img_size = img_size
        self.batch_size = batch_size
        self.class_mode = class_mode
        self.samples = []
        self.class_indices = {}
        classes = sorted(p.name for p in Path(directory).iterdir() if p.is_dir())
        for i, cls in enumerate(classes):
            self.class_indices[cls] = i
            for ext in ("*.jpg", "*.jpeg", "*.png", "*.bmp"):
                for fp in (Path(directory) / cls).glob(ext):
                    self.samples.append((str(fp), i))
        self.n = len(self.samples)
        self.classes = np.array([s[1] for s in self.samples])

    def __len__(self):
        return max(1, (self.n + self.batch_size - 1) // self.batch_size)

    def __getitem__(self, i):
        batch = self.samples[i * self.batch_size:(i + 1) * self.batch_size]
        imgs, labels = [], []
        for fp, label in batch:
            img = np.array(Image.open(fp).convert("RGB").resize(self.img_size))
            img = preprocess_input(img.astype(np.float32))
            imgs.append(img)
            labels.append(label)
        X = np.stack(imgs)
        if self.class_mode == "binary":
            Y = np.array(labels, dtype=np.float32)
        else:
            Y = np.eye(len(self.class_indices))[labels]
        return X, Y


In [ ]:
test1 = LeafSequence(
    SPLIT / "test" / "clasificacion_binaria",
    img_size=(240, 240), batch_size=32, class_mode="binary",
)
preds1 = (m1.predict(test1, verbose=1) > 0.5).astype(int).flatten()
reales1 = test1.classes[:len(preds1)]
print("=== Modelo 1 ===")
print(classification_report(reales1, preds1, target_names=list(test1.class_indices.keys()), digits=4))

fig, ax = plt.subplots(figsize=(5, 4))
cm1 = confusion_matrix(reales1, preds1)
sns.heatmap(cm1, annot=True, fmt="d", cmap="Blues",
            xticklabels=list(test1.class_indices.keys()),
            yticklabels=list(test1.class_indices.keys()), ax=ax)
ax.set_title("Modelo 1 - Confusion")
plt.tight_layout()
plt.savefig(OUT / "cm_m1.png", dpi=120)
plt.show()

In [ ]:
test2 = LeafSequence(
    SPLIT / "test" / "clasificacion_patogeno",
    batch_size=32, class_mode="categorical",
)
probs2 = m2.predict(test2, verbose=1)

thresholds_path = OUT / "thresholds.json"
if thresholds_path.exists():
    thresholds = json.loads(thresholds_path.read_text())
else:
    thresholds = {name: 0.5 for name in test2.class_indices}

names2 = list(test2.class_indices.keys())
threshold_vector = np.array([thresholds.get(name, 0.5) for name in names2])
preds_multilabel = (probs2 >= threshold_vector).astype(int)

true_multilabel = np.eye(len(names2))[test2.classes[:len(probs2)]].astype(int)

from sklearn.metrics import average_precision_score, hamming_loss

per_class_ap = {}
per_class_f1 = {}
for i, name in enumerate(names2):
    per_class_ap[name] = float(average_precision_score(true_multilabel[:, i], probs2[:, i]))
    per_class_f1[name] = float(f1_score(true_multilabel[:, i], preds_multilabel[:, i], zero_division=0))

mAP = float(np.mean(list(per_class_ap.values())))
f1_macro = float(np.mean(list(per_class_f1.values())))
hl = float(hamming_loss(true_multilabel, preds_multilabel))

print("=== Modelo 2 multi-label ===")
print(f"mAP: {mAP:.4f}  |  F1 macro: {f1_macro:.4f}  |  Hamming loss: {hl:.4f}")
print("\nPer-class:")
for name in names2:
    print(f"  {name:20s} AP={per_class_ap[name]:.3f}  F1={per_class_f1[name]:.3f}  thr={thresholds.get(name, 0.5):.3f}")

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(names2))
ax.bar(x - 0.2, [per_class_ap[n] for n in names2], width=0.4, label="AP")
ax.bar(x + 0.2, [per_class_f1[n] for n in names2], width=0.4, label="F1")
ax.set_xticks(x)
ax.set_xticklabels(names2, rotation=30)
ax.set_ylim(0, 1)
ax.legend()
ax.grid(alpha=0.3)
ax.set_title("Modelo 2 multi-label - AP vs F1 por clase")
plt.tight_layout()
plt.savefig(OUT / "m2_per_class_metrics.png", dpi=120)
plt.show()

In [ ]:
metrics = {
    "m1": {
        "accuracy": float(accuracy_score(reales1, preds1)),
        "f1": float(f1_score(reales1, preds1, zero_division=0)),
        "precision": float(precision_score(reales1, preds1, zero_division=0)),
        "recall": float(recall_score(reales1, preds1, zero_division=0)),
    },
    "m2_multilabel": {
        "mAP": mAP,
        "f1_macro": f1_macro,
        "hamming_loss": hl,
        "per_class_ap": per_class_ap,
        "per_class_f1": per_class_f1,
        "thresholds": thresholds,
    },
}
with open(OUT / "training_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)
print(json.dumps(metrics, indent=2, ensure_ascii=False))
print("\n=== Verificación de objetivos ===")
print(f"M1 accuracy: {metrics['m1']['accuracy']:.4f}  (objetivo >= 0.85) {'OK' if metrics['m1']['accuracy'] >= 0.85 else 'FALLA'}")
print(f"M1 F1:       {metrics['m1']['f1']:.4f}  (objetivo >= 0.85) {'OK' if metrics['m1']['f1'] >= 0.85 else 'FALLA'}")
print(f"M2 mAP:      {mAP:.4f}  (objetivo >= 0.85) {'OK' if mAP >= 0.85 else 'FALLA'}")
print(f"M2 macro F1: {f1_macro:.4f}  (objetivo >= 0.80) {'OK' if f1_macro >= 0.80 else 'FALLA'}")
for cls, f1v in per_class_f1.items():
    print(f"  {cls:20s}: F1={f1v:.4f}  {'OK' if f1v >= 0.70 else 'REVISAR'}")